# שלב 01 — טעינה וניקוי נתוני GTFS

**מטרה:** לטעון את קבצי ה‑GTFS הגולמיים של התחבורה הציבורית בישראל, לנקות תחנות עם קואורדינטות לא תקינות, ולשייך כל תחנה לאזור גיאוגרפי ולמטרופולין.

**קלט:** `israel-public-transportation/*.txt` (קבצי GTFS תקניים).  
**פלט:** `outputs/01_data_preparation/` — טבלאות מנוקות + דוח ניקוי.

למה זה חשוב: גרף שנבנה על דאטה לא נקייה יניב תוצאות מוטות. תחנה עם קואורדינטה שגויה תיצור קשתות מדומות ותקבל מרכזיות מלאכותית.

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas

## הגדרת נתיבים

התא הבא מאתר אוטומטית את שורש הפרויקט (לפי קיום תיקיית הנתונים), כך שה‑notebook ניתן להרצה ע"י צד שלישי בלי לשנות נתיבים.

In [ ]:
from pathlib import Path
import json
import math
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """מאתר את שורש הפרויקט לפי קיום תיקיית הנתונים israel-public-transportation."""
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
DATA_DIR = ROOT / "israel-public-transportation"
OUT_DIR = ROOT / "public_transport_network_notebooks" / "outputs" / "01_data_preparation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT     :", ROOT)
print("DATA_DIR :", DATA_DIR)
print("OUT_DIR  :", OUT_DIR)

## טעינת התחנות וניקוי קואורדינטות

טוענים את `stops.txt`, ממירים קו רוחב/אורך למספרים, ומסירים תחנות ללא קואורדינטות או כאלה שמחוץ לגבולות ישראל (רוחב 29–34, אורך 34–36). זה מסנן תחנות מדומות ושגיאות הזנה.

In [ ]:
def load_stops():
    df = pd.read_csv(DATA_DIR / "stops.txt", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    before = len(df)
    df["stop_lat"] = pd.to_numeric(df["stop_lat"], errors="coerce")
    df["stop_lon"] = pd.to_numeric(df["stop_lon"], errors="coerce")
    df = df.dropna(subset=["stop_lat", "stop_lon"])
    df = df[(df["stop_lat"] > 29) & (df["stop_lat"] < 34)]
    df = df[(df["stop_lon"] > 34) & (df["stop_lon"] < 36)]
    after = len(df)
    print(f"  stops: {before} -> {after} (removed {before-after} with bad coords)")
    return df


stops = load_stops()
stops.head()

## טעינת קווים, נסיעות ומפעילים

שלושת הקבצים הנוספים: `routes.txt` (קווים וסוגיהם), `trips.txt` (נסיעות, כל נסיעה שייכת לקו), ו‑`agency.txt` (מפעילים, כמו אגד/דן/מטרופולין).

In [ ]:
def load_routes():
    df = pd.read_csv(DATA_DIR / "routes.txt", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    print(f"  routes: {len(df)}")
    return df


def load_trips():
    df = pd.read_csv(DATA_DIR / "trips.txt", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    print(f"  trips: {len(df)}")
    return df


def load_agencies():
    df = pd.read_csv(DATA_DIR / "agency.txt", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    print(f"  agencies: {len(df)}")
    return df


routes = load_routes()
trips = load_trips()
agencies = load_agencies()

## שיוך אזור ומטרופולין לכל תחנה

כדי לאפשר השוואה אזורית בהמשך (שלב 06), משייכים כל תחנה ל‑**אזור** (צפון/מרכז/דרום/ירושלים) לפי קו רוחב, ול‑**מטרופולין** הקרוב (תל אביב/חיפה/ירושלים/באר שבע) לפי מרחק הברסיין, או 'פריפריה' אם רחוקה מכולם.

In [ ]:
def assign_region(lat, lon):
    if 31.70 <= lat <= 31.90 and 34.95 <= lon <= 35.30:
        return "ירושלים"
    elif lat > 32.50:
        return "צפון"
    elif lat >= 31.55:
        return "מרכז"
    else:
        return "דרום"


def haversine(la1, lo1, la2, lo2):
    R = 6371
    phi1, phi2 = math.radians(la1), math.radians(la2)
    dphi = math.radians(la2 - la1)
    dlam = math.radians(lo2 - lo1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2*R*math.asin(math.sqrt(a))


def assign_metro(lat, lon):
    metros = {
        "תל אביב": (32.0853, 34.7818, 30),
        "חיפה":    (32.7940, 34.9896, 25),
        "ירושלים": (31.7683, 35.2137, 20),
        "באר שבע": (31.2518, 34.7913, 25),
    }
    for city, (clat, clon, radius) in metros.items():
        if haversine(lat, lon, clat, clon) <= radius:
            return city
    return "פריפריה"


stops["region"] = stops.apply(lambda r: assign_region(r["stop_lat"], r["stop_lon"]), axis=1)
stops["metro"] = stops.apply(lambda r: assign_metro(r["stop_lat"], r["stop_lon"]), axis=1)
stops[["stop_id", "stop_name", "stop_lat", "stop_lon", "region", "metro"]].head()

## שמירת הפלטים ודוח ניקוי

שומרים את הטבלאות המנוקות ל‑CSV, ויוצרים דוח JSON עם ספירות מסכמות (תחנות לפי אזור/מטרופולין, סוגי קווים). הפלטים האלה הם הקלט לשלב 02 (בניית הגרף).

In [ ]:
stops.to_csv(OUT_DIR / "stops_clean.csv", index=False, encoding="utf-8-sig")
routes.to_csv(OUT_DIR / "routes_clean.csv", index=False, encoding="utf-8-sig")
trips.to_csv(OUT_DIR / "trips_clean.csv", index=False, encoding="utf-8-sig")
agencies.to_csv(OUT_DIR / "agencies_clean.csv", index=False, encoding="utf-8-sig")

report = {
    "stops_total": len(stops),
    "routes_total": len(routes),
    "trips_total": len(trips),
    "agencies_total": len(agencies),
    "region_counts": stops["region"].value_counts().to_dict(),
    "metro_counts": stops["metro"].value_counts().to_dict(),
    "route_types": routes["route_type"].value_counts().to_dict(),
}
with open(OUT_DIR / "data_cleaning_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("=== סיכום ניקוי ===")
for k, v in report.items():
    print(f"  {k}: {v}")
print()
print("פלטים נשמרו ב:", OUT_DIR)